# LLM-as-a-Judge

## Why It Exists

You've built an eval dataset. You have hundreds of question/answer pairs. Your RAG system generates answers. Now someone has to **grade them**.

The naive approach: hire human reviewers. A procurement SME reads each system answer, compares it to the gold answer, and scores it. This is accurate. It's also slow, expensive, and impossible to run continuously as you iterate on your system.

The automated approach: write deterministic rules. If the answer contains the correct number, score it 1. This works for exact-match facts — EMD amounts, deadlines, vendor names — but completely breaks down for anything requiring judgment. Is this answer *complete enough*? Is the *tone* appropriate for a committee report? Is this a *meaningful* hallucination or a trivial one? Rules can't answer those questions.

**LLM-as-a-Judge exists because evaluation often requires the same kind of language understanding that generation does.** You need something that can read, reason, and make judgment calls — at scale, automatically, repeatedly.

### How It Works

The core idea is simple: prompt an LLM to act as an evaluator.

```
SYSTEM: You are an expert procurement evaluation assistant. Your job is
to assess the quality of answers generated by a RAG system against
source documents.

USER:
Question: "Does Vendor B meet the minimum turnover requirement in
RFP-2024-IT-047?"

Retrieved context: "Section 3.1 — Eligibility: Vendors must demonstrate
a minimum annual turnover of ₹10 Crore for each of the last 3 financial
years. Vendor B's audited financials show turnovers of ₹11.2 Cr,
₹9.8 Cr, and ₹12.1 Cr for FY22, FY23, and FY24 respectively."

Gold answer: "No. Vendor B does not meet the requirement. Their FY23
turnover of ₹9.8 Cr falls below the ₹10 Cr threshold."

System answer: "Vendor B's turnover over the last 3 years averaged
₹11.03 Cr, which meets the minimum requirement."

Score this answer on faithfulness (1–5) and correctness (1–5).
Explain your reasoning before giving scores.
```

The judge LLM reads the question, the context, the gold answer, and the system's answer — and returns scores with reasoning.

You run this across your entire eval dataset. In minutes, you have scores on hundreds of answers that would have taken days to review manually.

---

## What LLM-as-a-Judge Evaluates Well

### Fluency and Coherence
Is the answer readable? Does it flow logically? Does it communicate clearly?

A rule can't tell you that *"Vendor A price ₹18.4L. 3-year. GST included."* is worse than a properly structured sentence. A judge model reads both and immediately identifies which is fit for a procurement committee report.

### Faithfulness to Context
The judge can read the retrieved passage and the generated answer side by side and assess whether the LLM stayed within the bounds of what the document said — catching added clauses, missing conditions, or subtle distortions.

> *System answer says "delivery within 45 days." Context says "45 business days."* A judge catches this. A string-match rule misses it entirely.

### Tone and Register Appropriateness
In procurement, an answer going to a tender evaluation committee needs to sound different from an answer going to a vendor helpdesk.

> *"Looks like Vendor C's pricing is actually pretty good value!"* — a judge flags this as inappropriately casual for a formal evaluation context. No rule catches it.

### Completeness
Did the answer address the full question, or only part of it?

> Question: *"Which vendors fail mandatory eligibility and why?"*
> System answer: *"Vendor D fails mandatory eligibility."*

The answer is factually correct but misses that Vendor F also failed, and gives no reasoning. A judge model identifies the omission. A correctness check against the gold answer might not — especially if the gold answer is verbose and the comparison is approximate.

### Comparative Evaluation (Pairwise Judging)
Instead of scoring one answer, you show the judge two answers from different system versions and ask: *which is better, and why?*

This is how teams evaluate model upgrades — run the old system and new system on the same eval set, then have the judge compare outputs pair by pair. This is often more reliable than absolute scoring because relative comparison is easier than calibrated rating.

---

## What LLM-as-a-Judge Evaluates Poorly

### Precise Numerical Verification
If the correct answer is ₹18,40,000 and the system says ₹18,04,000, a judge *might* catch it — or might read past the transposed digits and rate the answer as correct because the surrounding language sounds right.

For numbers, dates, and exact figures — **deterministic string matching is more reliable than a judge**.

### Legal and Domain-Specific Precision
A general judge model doesn't know that *"substantially compliant"* and *"technically compliant"* carry different legal weight in Indian public procurement. It will evaluate them as roughly synonymous and miss a meaningful error.

Domain-specific failures require domain-specific evaluators — either a fine-tuned judge, or a human SME.

### Detecting Subtle Omissions
A judge is good at catching *obvious* missing information. It's poor at catching the clause that *should* have been mentioned but wasn't — because the judge doesn't know what it doesn't know.

> The correct answer requires mentioning a penalty clause in Section 9.4. The system answer never mentions it. The judge, not knowing to look for Section 9.4, scores the answer as complete.

This is why your eval dataset gold answers matter so much — the judge compares against what you told it to look for, not against a comprehensive understanding of the entire document.

### Consistency at Scale
Ask the same judge to score the same answer on two different days, or in two different conversation positions, and you may get different scores. LLMs are probabilistic. At small scales this is noise; at large scales it becomes a calibration problem.

Mitigation: run each eval item multiple times and average scores, or use temperature=0 for determinism.

---

## Self-Preference Bias

This is the most important failure mode to understand when designing your evaluation pipeline.

### What It Is

When you ask an LLM to judge answers, it tends to rate outputs that **resemble its own writing style** more favourably — even when those outputs aren't objectively better.

This has been documented in research: models trained similarly, from the same base, or from the same family systematically prefer each other's outputs. The judge isn't consciously cheating. It's a pattern-matching system, and it recognises — and rewards — patterns it was trained to produce.

### What It Means in Practice

Imagine your setup:

- **Generator:** Claude Sonnet
- **Judge:** Claude Sonnet (same model)

The generator produces answers in a particular style — structured, hedged, with specific phrasing patterns. The judge has been trained on similar data and produces similar patterns. When it evaluates the generator's output, it's essentially asking: *does this look like something I would write?* The answer is frequently yes — so scores are inflated.

Now flip it:

- **Generator:** Claude Sonnet
- **Judge:** A GPT-family model

The judge has different stylistic tendencies, different hedging patterns, different structural preferences. The same answer that Claude-as-judge rated 4.5/5 might get 3.8/5 from a GPT-judge — not because the answer got worse, but because the stylistic resonance is gone.

### A Concrete Procurement Example

Generator produces:
> *"Based on the submitted documentation, Vendor A appears to meet the eligibility criteria as outlined in Section 3.1, though it would be advisable to verify the audited financials independently."*

A same-family judge reads this and scores it highly — it's well-hedged, appropriately cautious, structurally clean. These are patterns the judge itself produces.

A different-family judge might penalise the hedge as vague and prefer a direct: *"Vendor A meets the eligibility criteria per Section 3.1. FY22–FY24 turnovers of ₹11.2 Cr, ₹9.8 Cr, and ₹12.1 Cr exceed the ₹10 Cr threshold."*

Neither judge is objectively right about which answer is better. They're reflecting their own stylistic training.

### How Teams Mitigate This

**Use a different model family as judge than as generator.** If you generate with Claude, judge with GPT or Gemini, and vice versa. This doesn't eliminate bias but breaks the same-family amplification loop.

**Use multiple judges and aggregate.** Run two or three judge models, average their scores. Disagreements surface where one model's bias is distorting the picture.

**Anchor with rubrics, not open-ended scoring.** Instead of asking *"Is this a good answer? Score 1–5,"* give the judge explicit criteria:
- Does the answer cite the correct section number? (Yes/No)
- Does the answer contain any claim not present in the context? (Yes/No)
- Does the answer address all parts of the question? (Yes/No/Partial)

Structured rubrics constrain the judge's latitude and reduce how much stylistic preference can influence the score.

**Validate your judge against human scores periodically.** Pick 50 answers, have a human score them, compare to judge scores. If judge scores are systematically higher than human scores across the board — self-preference bias is likely inflating results.

---

## When Human Review Is Still Necessary

LLM-as-a-Judge is a **force multiplier**, not a replacement. There are specific situations where human review is non-negotiable.

### 1. Calibrating the Judge Itself

Before you trust your judge, you have to verify it. That requires humans scoring the same items and comparing. If the judge disagrees with humans 40% of the time, you have a bad judge — and you'd never know without human validation.

This calibration exercise should be repeated periodically, especially after system changes. A judge that was well-calibrated against your old system may drift when your generator or retrieval changes.

### 2. High-Stakes Irreversible Decisions

A procurement committee is about to disqualify a vendor based on a RAG system answer. The judge scored the system's answer as correct and complete.

This should still go to a human. LLM judges — like all LLMs — fail silently. They don't tell you when they're wrong with confidence. In decisions that are difficult to reverse — contract awards, disqualifications, penalty triggers — the cost of a confident wrong answer is too high to delegate entirely to an automated judge.

### 3. Novel Failure Modes

Your judge is only as good as what it's been asked to evaluate. When you change your system — new chunking strategy, new model, new document types — new failure modes emerge that your existing judge prompts may not be designed to detect.

Humans reviewing a sample of outputs after a system change will spot failure patterns that your automated pipeline simply isn't looking for yet. You then encode those patterns into the judge prompt.

> Human review is how you discover what to add to your rubric. The judge then scales that discovery across thousands of queries.

### 4. Regulatory or Audit Contexts

In public procurement in India, GeM or CPPP guidelines may require human sign-off on evaluations. An LLM judge score is not a defensible audit trail. When accountability matters — when someone has to sign their name to a decision — humans remain in the loop.

### 5. Disagreement Between Judges

When you run multiple judge models and they disagree significantly on a case, that disagreement is a signal: this item is genuinely ambiguous or difficult. Averaging the scores hides the problem. A human should look at exactly these cases, because they represent either a failure the judges can't characterise, or a genuine edge case your rubric doesn't cover.

---

## How It All Fits Together

```
Eval Dataset (Q&A pairs)
         │
         ▼
RAG System generates answers
         │
         ▼
LLM Judge scores each answer
  ├── Faithfulness
  ├── Completeness
  ├── Tone / Register
  └── Correctness vs. gold answer
         │
         ▼
Automated scores → track across system versions
         │
         ├── Spot-check 5–10% with humans
         │     └── Recalibrate judge if drift detected
         │
         ├── Flag low-confidence / judge-disagreement items
         │     └── Send to human review
         │
         └── High-stakes outputs
               └── Always to human before action
```

The honest summary: **LLM-as-a-Judge is good at scale, poor at precision, and blind to its own biases.** The teams that use it well treat judge scores as a strong signal, not a ground truth — and keep humans close enough to catch what the judge cannot see.